# LANL Cyber 1: behavioral feature engineering

This notebook creates causal, time-windowed authentication features for a small hackathon demo. It prepares a feature matrix but does not train Isolation Forest or any other model.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_auth_events_window, load_redteam_events
from src.features import FEATURE_COLUMNS, WINDOW_SECONDS, build_feature_dataset

AUTH_PATH = PROJECT_ROOT / 'data' / 'raw' / 'auth.txt.gz'
REDTEAM_PATH = PROJECT_ROOT / 'data' / 'raw' / 'redteam.txt.gz'
REDTEAM_SAMPLE_SIZE = 15
BEFORE = WINDOW_SECONDS
AFTER = WINDOW_SECONDS

redteam = load_redteam_events(REDTEAM_PATH, max_rows=REDTEAM_SAMPLE_SIZE)
display(redteam)

,timestamp,user,source_computer,destination_computer
0,150885,U620@DOM1,C17693,C1003
1,151036,U748@DOM1,C17693,C305
2,151648,U748@DOM1,C17693,C728
3,151993,U6115@DOM1,C17693,C1173
4,153792,U636@DOM1,C17693,C294
5,155219,U748@DOM1,C17693,C5693
6,155399,U748@DOM1,C17693,C152
7,155460,U748@DOM1,C17693,C2341
8,155591,U748@DOM1,C17693,C332
9,156658,U748@DOM1,C17693,C4280


## Bounded authentication input

We stream the auth gzip file and retain only the timestamp range around the selected red-team events. This is a manageable in-memory subset, not the complete 7+ GB archive. Red-team rows are used only to choose an inspection range and comparison timestamps, never as feature inputs.

In [2]:
auth_start = int(redteam['timestamp'].min()) - BEFORE
auth_end = int(redteam['timestamp'].max()) + AFTER
auth_events = load_auth_events_window(AUTH_PATH, auth_start, auth_end)

redteam_entities = set(redteam['user']) | set(redteam['source_computer']) | set(redteam['destination_computer'])
feature_timestamps = sorted(set(int(value) for value in redteam['timestamp']))
print('Auth rows retained:', len(auth_events))
print('Auth timestamp range:', int(auth_events['timestamp'].min()), 'to', int(auth_events['timestamp'].max()))
print('Entities used for this demo:', len(redteam_entities))
print('Feature timestamps:', feature_timestamps)

Auth rows retained: 14086748
Auth timestamp range: 150585 to 227352
Entities used for this demo: 19
Feature timestamps: [150885, 151036, 151648, 151993, 153792, 155219, 155399, 155460, 155591, 156658, 210086, 210294, 210312, 218418, 227052]


## Causal feature definition

Each row represents one entity at an event timestamp. Its active window is `[timestamp - 300 + 1, timestamp]`. Novel destinations and edges are compared only with events strictly earlier than the active window, so future observations cannot influence the row. A red-team label is not present in the feature vector.

In [3]:
features = build_feature_dataset(
    auth_events,
    window_seconds=WINDOW_SECONDS,
    entities=redteam_entities,
    timestamps=feature_timestamps,
)
display(features.head())
print('Feature dataset shape:', features.shape)

,timestamp,entity,entity_type,window_start,window_end,total_auth_events,successful_auth_count,failed_auth_count,unique_source_computers,unique_destination_computers,new_destination_count,unique_users,new_edge_count,outgoing_degree,incoming_degree,event_rate
0,150885,C1003,COMPUTER,150586,150885,5,5,0,2,1,1,5,1,1,2,1.0
1,150885,C1173,COMPUTER,150586,150885,3,3,0,2,1,1,2,1,1,2,0.6
2,150885,C148,COMPUTER,150586,150885,1,1,0,1,1,1,1,1,1,0,0.2
3,150885,C1493,USER_OR_COMPUTER,150586,150885,0,0,0,0,0,0,0,0,0,0,0.0
4,150885,C152,USER_OR_COMPUTER,150586,150885,0,0,0,0,0,0,0,0,0,0,0.0


Feature dataset shape: (285, 16)


In [4]:
print('Feature distributions and statistics:')
display(features[FEATURE_COLUMNS].describe().T)
print('Missing values:')
display(features.isna().sum().rename('missing_count').to_frame())
print('Final feature columns:')
print(FEATURE_COLUMNS)

Feature distributions and statistics:


,count,mean,std,min,25%,50%,75%,max
total_auth_events,285.0,4.747368,14.699311,0.0,0.0,1.0,4.0,207.0
successful_auth_count,285.0,4.743860,14.699849,0.0,0.0,1.0,4.0,207.0
failed_auth_count,285.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
unique_source_computers,285.0,1.245614,1.848874,0.0,0.0,1.0,2.0,20.0
unique_destination_computers,285.0,1.364912,1.998282,0.0,0.0,1.0,2.0,20.0
new_destination_count,285.0,0.421053,0.966833,0.0,0.0,0.0,0.0,5.0
unique_users,285.0,1.052632,1.259066,0.0,0.0,1.0,1.0,7.0
new_edge_count,285.0,0.421053,0.966833,0.0,0.0,0.0,0.0,5.0
outgoing_degree,285.0,1.336842,1.989110,0.0,0.0,1.0,2.0,20.0
incoming_degree,285.0,0.638596,1.112945,0.0,0.0,0.0,1.0,6.0


Missing values:


,missing_count
timestamp,0
entity,0
entity_type,0
window_start,0
window_end,0
total_auth_events,0
successful_auth_count,0
failed_auth_count,0
unique_source_computers,0
unique_destination_computers,0


Final feature columns:
['total_auth_events', 'successful_auth_count', 'failed_auth_count', 'unique_source_computers', 'unique_destination_computers', 'new_destination_count', 'unique_users', 'new_edge_count', 'outgoing_degree', 'incoming_degree', 'event_rate']


## Compare feature values at known red-team timestamps

This comparison is for interpretation and later evaluation only. The known red-team events are not included in `X`.

In [5]:
redteam_comparison = features.merge(
    redteam[['timestamp', 'user', 'source_computer', 'destination_computer']],
    on='timestamp',
    how='inner',
    suffixes=('', '_redteam'),
)
display(redteam_comparison[['timestamp', 'entity', 'entity_type'] + FEATURE_COLUMNS].head(20))

high_activity = features.sort_values(
    ['new_destination_count', 'new_edge_count', 'event_rate'],
    ascending=False,
)
print('Rows with the strongest lateral-movement-oriented signals:')
display(high_activity[['timestamp', 'entity', 'entity_type'] + FEATURE_COLUMNS].head(10))

,timestamp,entity,entity_type,total_auth_events,successful_auth_count,failed_auth_count,unique_source_computers,unique_destination_computers,new_destination_count,unique_users,new_edge_count,outgoing_degree,incoming_degree,event_rate
0,150885,C1003,COMPUTER,5,5,0,2,1,1,5,1,1,2,1.0
1,150885,C1173,COMPUTER,3,3,0,2,1,1,2,1,1,2,0.6
2,150885,C148,COMPUTER,1,1,0,1,1,1,1,1,1,0,0.2
3,150885,C1493,USER_OR_COMPUTER,0,0,0,0,0,0,0,0,0,0,0.0
4,150885,C152,USER_OR_COMPUTER,0,0,0,0,0,0,0,0,0,0,0.0
5,150885,C17693,COMPUTER,3,3,0,1,2,2,1,2,2,0,0.6
6,150885,C18025,USER_OR_COMPUTER,0,0,0,0,0,0,0,0,0,0,0.0
7,150885,C2341,USER_OR_COMPUTER,0,0,0,0,0,0,0,0,0,0,0.0
8,150885,C294,COMPUTER,5,5,0,1,2,2,2,2,2,0,1.0
9,150885,C305,COMPUTER,1,1,0,1,1,1,1,1,1,0,0.2


Rows with the strongest lateral-movement-oriented signals:


,timestamp,entity,entity_type,total_auth_events,successful_auth_count,failed_auth_count,unique_source_computers,unique_destination_computers,new_destination_count,unique_users,new_edge_count,outgoing_degree,incoming_degree,event_rate
68,151993,C4280,COMPUTER,26,26,0,2,6,5,1,5,6,2,5.2
52,151648,C728,COMPUTER,24,24,0,3,6,5,2,5,6,3,4.8
129,155399,U6115@DOM1,USER,24,24,0,6,5,5,1,5,5,1,4.8
148,155460,U6115@DOM1,USER,24,24,0,6,5,5,1,5,5,1,4.8
208,210086,U748@DOM1,USER,207,207,0,20,20,4,1,4,20,1,41.4
79,153792,C1493,COMPUTER,14,14,0,1,5,4,2,4,5,1,2.8
56,151648,U748@DOM1,USER,12,12,0,7,6,4,1,4,6,1,2.4
37,151036,U748@DOM1,USER,8,8,0,4,4,4,1,4,4,1,1.6
170,155591,U748@DOM1,USER,21,21,0,6,5,3,1,3,5,1,4.2
151,155460,U748@DOM1,USER,16,16,0,5,4,3,1,3,4,1,3.2


## Prepare the next-stage feature matrix

`timestamp`, `entity`, `entity_type`, and window boundaries are context or identifiers, so they are excluded from `X`. The count and rate features are already numeric and on comparable small count/rate scales for this demo. Scaling is not required by Isolation Forest, though it may be useful for models based on distance or gradient optimization. No model is trained here.

In [6]:
X = features[FEATURE_COLUMNS].copy()
print('X.shape:', X.shape)
print('Feature names:', list(X.columns))
print('Missing values in X:')
display(X.isna().sum().rename('missing_count').to_frame())
print('Basic statistics for X:')
display(X.describe().T)
print('Scaling necessary for Isolation Forest? No, not required for this feature set.')

X.shape: (285, 11)
Feature names: ['total_auth_events', 'successful_auth_count', 'failed_auth_count', 'unique_source_computers', 'unique_destination_computers', 'new_destination_count', 'unique_users', 'new_edge_count', 'outgoing_degree', 'incoming_degree', 'event_rate']
Missing values in X:


,missing_count
total_auth_events,0
successful_auth_count,0
failed_auth_count,0
unique_source_computers,0
unique_destination_computers,0
new_destination_count,0
unique_users,0
new_edge_count,0
outgoing_degree,0
incoming_degree,0


Basic statistics for X:


,count,mean,std,min,25%,50%,75%,max
total_auth_events,285.0,4.747368,14.699311,0.0,0.0,1.0,4.0,207.0
successful_auth_count,285.0,4.743860,14.699849,0.0,0.0,1.0,4.0,207.0
failed_auth_count,285.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
unique_source_computers,285.0,1.245614,1.848874,0.0,0.0,1.0,2.0,20.0
unique_destination_computers,285.0,1.364912,1.998282,0.0,0.0,1.0,2.0,20.0
new_destination_count,285.0,0.421053,0.966833,0.0,0.0,0.0,0.0,5.0
unique_users,285.0,1.052632,1.259066,0.0,0.0,1.0,1.0,7.0
new_edge_count,285.0,0.421053,0.966833,0.0,0.0,0.0,0.0,5.0
outgoing_degree,285.0,1.336842,1.989110,0.0,0.0,1.0,2.0,20.0
incoming_degree,285.0,0.638596,1.112945,0.0,0.0,0.0,1.0,6.0


Scaling necessary for Isolation Forest? No, not required for this feature set.


In [7]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / 'features_sample.parquet'
features.to_parquet(output_path, index=False)
print('Saved:', output_path)

Saved: d:\Hackathons\SOC Analyst\autonomous-threat-defense\data\processed\features_sample.parquet
